In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 데이터 불러오기

In [2]:
train_df = pd.read_csv('data/titanic_train3.csv')
test_df = pd.read_csv('data/titanic_test3.csv')

In [3]:
train_df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,isChild,FamilySize,FamilySizeGroup
0,0,3,male,22.0,1,0,2.110213,S,Mr,0.0,1,1
1,1,1,female,38.0,1,0,4.280593,C,Mrs,0.0,1,1
2,1,3,female,26.0,0,0,2.188856,S,Miss,0.0,0,0
3,1,1,female,35.0,1,0,3.990834,S,Mrs,0.0,1,1
4,0,3,male,35.0,0,0,2.202765,S,Mr,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,2.639057,S,Rare,0.0,0,0
887,1,1,female,19.0,0,0,3.433987,S,Miss,0.0,0,0
888,0,3,female,28.0,1,2,3.196630,S,Miss,0.0,3,1
889,1,1,male,26.0,0,0,3.433987,C,Mr,0.0,0,0


In [4]:
test_df

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,isChild,FamilySize,FamilySizeGroup
0,3,male,34.5,0,0,2.178064,Q,Mr,0.0,0,0
1,3,female,47.0,1,0,2.079442,S,Mrs,0.0,1,1
2,2,male,62.0,0,0,2.369075,Q,Mr,0.0,0,0
3,3,male,27.0,0,0,2.268252,S,Mr,0.0,0,0
4,3,female,22.0,1,1,2.586824,S,Mrs,0.0,2,1
...,...,...,...,...,...,...,...,...,...,...,...
413,3,male,28.0,0,0,2.202765,S,Mr,0.0,0,0
414,1,female,39.0,0,0,4.699571,C,Rare,0.0,0,0
415,3,male,38.5,0,0,2.110213,S,Mr,0.0,0,0
416,3,male,28.0,0,0,2.202765,S,Mr,0.0,0,0


### Pclass

In [5]:
train_df['Pclass'].value_counts()

Pclass
3    491
1    216
2    184
Name: count, dtype: int64

In [6]:
# 값의 종류가 3가지 이므로 OneHotEncoder를 사용한다.
encoder = OneHotEncoder()
# 학습
encoder.fit(train_df[['Pclass']])
# 변환
pclass_train_encoded = encoder.transform(train_df[['Pclass']]).toarray()
pclass_test_encoded = encoder.transform(test_df[['Pclass']]).toarray()

# 새로운 컬럼 이름
columns = encoder.get_feature_names_out(['Pclass'])

# 데이터 프레임을 생성한다.
encoded_train_df = pd.DataFrame(pclass_train_encoded, columns=columns)
encoded_test_df = pd.DataFrame(pclass_test_encoded, columns=columns)

# 원본에 붙여준다.
train_df = pd.concat([train_df, encoded_train_df], axis=1)
test_df = pd.concat([test_df, encoded_test_df], axis=1)

# Pclass 컬럼을 삭제한다
train_df.drop('Pclass', axis=1, inplace=True)
test_df.drop('Pclass', axis=1, inplace=True)

display(train_df)
display(test_df)

,Survived,Sex,Age,SibSp,Parch,Fare,Embarked,Title,isChild,FamilySize,FamilySizeGroup,Pclass_1,Pclass_2,Pclass_3
0,0,male,22.0,1,0,2.110213,S,Mr,0.0,1,1,0.0,0.0,1.0
1,1,female,38.0,1,0,4.280593,C,Mrs,0.0,1,1,1.0,0.0,0.0
2,1,female,26.0,0,0,2.188856,S,Miss,0.0,0,0,0.0,0.0,1.0
3,1,female,35.0,1,0,3.990834,S,Mrs,0.0,1,1,1.0,0.0,0.0
4,0,male,35.0,0,0,2.202765,S,Mr,0.0,0,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,male,27.0,0,0,2.639057,S,Rare,0.0,0,0,0.0,1.0,0.0
887,1,female,19.0,0,0,3.433987,S,Miss,0.0,0,0,1.0,0.0,0.0
888,0,female,28.0,1,2,3.196630,S,Miss,0.0,3,1,0.0,0.0,1.0
889,1,male,26.0,0,0,3.433987,C,Mr,0.0,0,0,1.0,0.0,0.0


,Sex,Age,SibSp,Parch,Fare,Embarked,Title,isChild,FamilySize,FamilySizeGroup,Pclass_1,Pclass_2,Pclass_3
0,male,34.5,0,0,2.178064,Q,Mr,0.0,0,0,0.0,0.0,1.0
1,female,47.0,1,0,2.079442,S,Mrs,0.0,1,1,0.0,0.0,1.0
2,male,62.0,0,0,2.369075,Q,Mr,0.0,0,0,0.0,1.0,0.0
3,male,27.0,0,0,2.268252,S,Mr,0.0,0,0,0.0,0.0,1.0
4,female,22.0,1,1,2.586824,S,Mrs,0.0,2,1,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,male,28.0,0,0,2.202765,S,Mr,0.0,0,0,0.0,0.0,1.0
414,female,39.0,0,0,4.699571,C,Rare,0.0,0,0,1.0,0.0,0.0
415,male,38.5,0,0,2.110213,S,Mr,0.0,0,0,0.0,0.0,1.0
416,male,28.0,0,0,2.202765,S,Mr,0.0,0,0,0.0,0.0,1.0


In [7]:
# 값의 종류를 확인한다
train_df['Sex'].value_counts()

Sex
male      577
female    314
Name: count, dtype: int64

In [8]:
# 값의 종류가 두 가지 이므로 LabelEncoder를 사용한다.
encoder = LabelEncoder()
encoder.fit(train_df['Sex'])

train_df['Sex'] = encoder.transform(train_df['Sex'])
test_df['Sex'] = encoder.transform(test_df['Sex'])

display(train_df)
display(test_df)

,Survived,Sex,Age,SibSp,Parch,Fare,Embarked,Title,isChild,FamilySize,FamilySizeGroup,Pclass_1,Pclass_2,Pclass_3
0,0,1,22.0,1,0,2.110213,S,Mr,0.0,1,1,0.0,0.0,1.0
1,1,0,38.0,1,0,4.280593,C,Mrs,0.0,1,1,1.0,0.0,0.0
2,1,0,26.0,0,0,2.188856,S,Miss,0.0,0,0,0.0,0.0,1.0
3,1,0,35.0,1,0,3.990834,S,Mrs,0.0,1,1,1.0,0.0,0.0
4,0,1,35.0,0,0,2.202765,S,Mr,0.0,0,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,1,27.0,0,0,2.639057,S,Rare,0.0,0,0,0.0,1.0,0.0
887,1,0,19.0,0,0,3.433987,S,Miss,0.0,0,0,1.0,0.0,0.0
888,0,0,28.0,1,2,3.196630,S,Miss,0.0,3,1,0.0,0.0,1.0
889,1,1,26.0,0,0,3.433987,C,Mr,0.0,0,0,1.0,0.0,0.0


,Sex,Age,SibSp,Parch,Fare,Embarked,Title,isChild,FamilySize,FamilySizeGroup,Pclass_1,Pclass_2,Pclass_3
0,1,34.5,0,0,2.178064,Q,Mr,0.0,0,0,0.0,0.0,1.0
1,0,47.0,1,0,2.079442,S,Mrs,0.0,1,1,0.0,0.0,1.0
2,1,62.0,0,0,2.369075,Q,Mr,0.0,0,0,0.0,1.0,0.0
3,1,27.0,0,0,2.268252,S,Mr,0.0,0,0,0.0,0.0,1.0
4,0,22.0,1,1,2.586824,S,Mrs,0.0,2,1,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1,28.0,0,0,2.202765,S,Mr,0.0,0,0,0.0,0.0,1.0
414,0,39.0,0,0,4.699571,C,Rare,0.0,0,0,1.0,0.0,0.0
415,1,38.5,0,0,2.110213,S,Mr,0.0,0,0,0.0,0.0,1.0
416,1,28.0,0,0,2.202765,S,Mr,0.0,0,0,0.0,0.0,1.0


### Embarked

In [9]:
train_df['Embarked'].value_counts()

Embarked
S    646
C    168
Q     77
Name: count, dtype: int64

In [10]:
# 값의 종류가 3가지 이므로 OneHotEncoder를 사용한다.
encoder = OneHotEncoder()
encoder.fit(train_df[['Embarked']])

embarked_train_encoded = encoder.transform(train_df[['Embarked']]).toarray()
embarked_test_encoded = encoder.transform(test_df[['Embarked']]).toarray()

columns = encoder.get_feature_names_out(['Embarked'])

train_encoded_df = pd.DataFrame(embarked_train_encoded, columns = columns)
test_encoded_df = pd.DataFrame(embarked_test_encoded, columns=columns)

train_df = pd.concat([train_df,train_encoded_df], axis=1)
test_df = pd.concat([test_df, test_encoded_df], axis=1)

train_df.drop('Embarked',axis=1,inplace=True)
test_df.drop('Embarked',axis=1,inplace=True)

display(train_df)
display(test_df)

,Survived,Sex,Age,SibSp,Parch,Fare,Title,isChild,FamilySize,FamilySizeGroup,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Embarked_Q,Embarked_S
0,0,1,22.0,1,0,2.110213,Mr,0.0,1,1,0.0,0.0,1.0,0.0,0.0,1.0
1,1,0,38.0,1,0,4.280593,Mrs,0.0,1,1,1.0,0.0,0.0,1.0,0.0,0.0
2,1,0,26.0,0,0,2.188856,Miss,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0
3,1,0,35.0,1,0,3.990834,Mrs,0.0,1,1,1.0,0.0,0.0,0.0,0.0,1.0
4,0,1,35.0,0,0,2.202765,Mr,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,1,27.0,0,0,2.639057,Rare,0.0,0,0,0.0,1.0,0.0,0.0,0.0,1.0
887,1,0,19.0,0,0,3.433987,Miss,0.0,0,0,1.0,0.0,0.0,0.0,0.0,1.0
888,0,0,28.0,1,2,3.196630,Miss,0.0,3,1,0.0,0.0,1.0,0.0,0.0,1.0
889,1,1,26.0,0,0,3.433987,Mr,0.0,0,0,1.0,0.0,0.0,1.0,0.0,0.0


,Sex,Age,SibSp,Parch,Fare,Title,isChild,FamilySize,FamilySizeGroup,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Embarked_Q,Embarked_S
0,1,34.5,0,0,2.178064,Mr,0.0,0,0,0.0,0.0,1.0,0.0,1.0,0.0
1,0,47.0,1,0,2.079442,Mrs,0.0,1,1,0.0,0.0,1.0,0.0,0.0,1.0
2,1,62.0,0,0,2.369075,Mr,0.0,0,0,0.0,1.0,0.0,0.0,1.0,0.0
3,1,27.0,0,0,2.268252,Mr,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0
4,0,22.0,1,1,2.586824,Mrs,0.0,2,1,0.0,0.0,1.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1,28.0,0,0,2.202765,Mr,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0
414,0,39.0,0,0,4.699571,Rare,0.0,0,0,1.0,0.0,0.0,1.0,0.0,0.0
415,1,38.5,0,0,2.110213,Mr,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0
416,1,28.0,0,0,2.202765,Mr,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0


### Title

In [11]:
train_df['Title'].value_counts()

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [12]:
# 3가지 이상이므로 OneHotEncoder를 사용한다
encoder = OneHotEncoder()

encoder.fit(train_df[['Title']])

title_train_encoded = encoder.transform(train_df[['Title']]).toarray()
title_test_encoded = encoder.transform(test_df[['Title']]).toarray()

columns = encoder.get_feature_names_out(['Title'])

train_encoded_df = pd.DataFrame(title_train_encoded, columns=columns)
test_encoded_df = pd.DataFrame(title_test_encoded, columns=columns)

train_df = pd.concat([train_df, train_encoded_df], axis=1)
test_df = pd.concat([test_df,test_encoded_df], axis=1)

train_df.drop('Title', axis=1, inplace=True)
test_df.drop('Title', axis=1, inplace=True)

display(train_df)
display(test_df)

,Survived,Sex,Age,SibSp,Parch,Fare,isChild,FamilySize,FamilySizeGroup,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,1,22.0,1,0,2.110213,0.0,1,1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,1,0,38.0,1,0,4.280593,0.0,1,1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1,0,26.0,0,0,2.188856,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3,1,0,35.0,1,0,3.990834,0.0,1,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0,1,35.0,0,0,2.202765,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,1,27.0,0,0,2.639057,0.0,0,0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
887,1,0,19.0,0,0,3.433987,0.0,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
888,0,0,28.0,1,2,3.196630,0.0,3,1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
889,1,1,26.0,0,0,3.433987,0.0,0,0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


,Sex,Age,SibSp,Parch,Fare,isChild,FamilySize,FamilySizeGroup,Pclass_1,Pclass_2,Pclass_3,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,1,34.5,0,0,2.178064,0.0,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0,47.0,1,0,2.079442,0.0,1,1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,1,62.0,0,0,2.369075,0.0,0,0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,1,27.0,0,0,2.268252,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0,22.0,1,1,2.586824,0.0,2,1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1,28.0,0,0,2.202765,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
414,0,39.0,0,0,4.699571,0.0,0,0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
415,1,38.5,0,0,2.110213,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
416,1,28.0,0,0,2.202765,0.0,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


### isChild

In [13]:
# 값 종류를 확인한다
train_df['isChild'].value_counts()

isChild
0.0    820
1.0     71
Name: count, dtype: int64

- 값이 두 가지이며 0, 1로 구성되어 있으므로 처리하지 않는다

### FamilySizeGroup

In [14]:
# 값의 종류를 확인한다.
train_df['FamilySizeGroup'].value_counts()

FamilySizeGroup
0    537
1    292
2     62
Name: count, dtype: int64

In [15]:
# 0부터 1씩 증가하는 것으로 되어 있지만 값의 종류가 3가지 이므로 OneHotEncoder를 사용한다.
encoder = OneHotEncoder()

encoder.fit(train_df[['FamilySizeGroup']])

group_train_encoded = encoder.transform(train_df[['FamilySizeGroup']]).toarray()
group_test_encoded = encoder.transform(test_df[['FamilySizeGroup']]).toarray()

columns = encoder.get_feature_names_out(['FamilySizeGroup'])

train_encoded_df = pd.DataFrame(group_train_encoded, columns=columns)
test_encoded_df = pd.DataFrame(group_test_encoded, columns=columns)

train_df = pd.concat([train_df,train_encoded_df], axis=1)
test_df = pd.concat([test_df, test_encoded_df], axis=1)

train_df.drop('FamilySizeGroup', axis=1, inplace=True)
test_df.drop('FamilySizeGroup', axis=1, inplace=True)

display(train_df)
display(test_df)

,Survived,Sex,Age,SibSp,Parch,Fare,isChild,FamilySize,Pclass_1,Pclass_2,...,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,FamilySizeGroup_0,FamilySizeGroup_1,FamilySizeGroup_2
0,0,1,22.0,1,0,2.110213,0.0,1,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,1,0,38.0,1,0,4.280593,0.0,1,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,1,0,26.0,0,0,2.188856,0.0,0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,1,0,35.0,1,0,3.990834,0.0,1,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,0,1,35.0,0,0,2.202765,0.0,0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,1,27.0,0,0,2.639057,0.0,0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
887,1,0,19.0,0,0,3.433987,0.0,0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
888,0,0,28.0,1,2,3.196630,0.0,3,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
889,1,1,26.0,0,0,3.433987,0.0,0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


,Sex,Age,SibSp,Parch,Fare,isChild,FamilySize,Pclass_1,Pclass_2,Pclass_3,...,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,FamilySizeGroup_0,FamilySizeGroup_1,FamilySizeGroup_2
0,1,34.5,0,0,2.178064,0.0,0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0,47.0,1,0,2.079442,0.0,1,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,1,62.0,0,0,2.369075,0.0,0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,1,27.0,0,0,2.268252,0.0,0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0,22.0,1,1,2.586824,0.0,2,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1,28.0,0,0,2.202765,0.0,0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
414,0,39.0,0,0,4.699571,0.0,0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
415,1,38.5,0,0,2.110213,0.0,0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
416,1,28.0,0,0,2.202765,0.0,0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


### 완료 되었으므로 저장

In [16]:
train_df.to_csv('data/titanic_train4.csv',index=False)
test_df.to_csv('data/titanic_test4.csv', index=False)